# Data Loading

In [1]:
import pandas as pd
import numpy as np 
import json

# load the raw JSON data
with open("../data/raw_credit_applications.json") as file:
    raw_data = json.load(file)

# flatten the standard nested objects and display the new column names
df = pd.json_normalize(raw_data)
print(df.columns)

Index(['_id', 'spending_behavior', 'processing_timestamp',
       'applicant_info.full_name', 'applicant_info.email',
       'applicant_info.ssn', 'applicant_info.ip_address',
       'applicant_info.gender', 'applicant_info.date_of_birth',
       'applicant_info.zip_code', 'financials.annual_income',
       'financials.credit_history_months', 'financials.debt_to_income',
       'financials.savings_balance', 'decision.loan_approved',
       'decision.rejection_reason', 'loan_purpose', 'decision.interest_rate',
       'decision.approved_amount', 'financials.annual_salary', 'notes'],
      dtype='object')


# Data Handling

In [2]:
# handle the spending_behavior array of dictionaries 
spending_exploded = df[['_id', 'spending_behavior']].explode('spending_behavior').dropna(subset=['spending_behavior'])

# flatten these new exploded column since it includes category and amount
spending_normalize = pd.json_normalize(spending_exploded['spending_behavior'])
spending_normalize.insert(0, '_id', spending_exploded['_id'].values)

# pivot the data to have one row per id with spending categories as columns
spending_pivot = spending_normalize.pivot_table(
    index='_id',
    columns='category',
    values='amount',
    fill_value=0
).add_prefix('spending_')

# join these new spending columns with the main df and drop the old raw array
df_clean = df.drop(columns=['spending_behavior'])
df_clean = df_clean.merge(spending_pivot, on='_id', how='left')

display(df_clean.head())

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,...,spending_Fitness,spending_Gambling,spending_Groceries,spending_Healthcare,spending_Insurance,spending_Rent,spending_Shopping,spending_Transportation,spending_Travel,spending_Utilities
0,app_200,2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,...,0.0,0.0,0.0,0.0,0.0,790.0,480.0,0.0,0.0,0.0
1,app_037,NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,...,0.0,0.0,0.0,243.0,0.0,608.0,0.0,0.0,0.0,0.0
2,app_215,NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,...,0.0,0.0,0.0,0.0,0.0,109.0,0.0,0.0,0.0,0.0
3,app_024,NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000,...,575.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,app_184,2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,57000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Intentional Data Issues Assessment

### 1. Duplicate Records 

In [3]:
# find exact duplicates and count them
duplicates = df_clean[df_clean.duplicated(subset=['_id'], keep=False)]
duplicate_count = df_clean.duplicated(subset=['_id']).sum()
duplicate_percentage = (duplicate_count / len(df_clean)) * 100

# display the duplicates 
print(f'Found {duplicate_count} duplicated applications, which represent {duplicate_percentage:.2f}% of the total:')
display(duplicates[['_id', 'applicant_info.full_name', 'applicant_info.email']])

Found 2 duplicated applications, which represent 0.40% of the total:


,_id,applicant_info.full_name,applicant_info.email
8,app_042,Joseph Lopez,joseph.lopez1@gmail.com
354,app_042,Joseph Lopez,joseph.lopez1@gmail.com
383,app_001,Stephanie Nguyen,stephanie.nguyen47@mail.com
455,app_001,Stephanie Nguyen,stephanie.nguyen47@mail.com


### 2. Inconsistent data types across records

In [4]:
# check if all the fields that should be numerical are actually numeric
expected_numeric = [
    'financials.annual_income', 
    'financials.annual_salary', 
    'financials.credit_history_months', 
    'financials.debt_to_income', 
    'financials.savings_balance',
    'decision.interest_rate',
    'decision.approved_amount'
]

for col in expected_numeric:
    valid_rows = df_clean[df_clean[col].notnull()]
    string_count = valid_rows[col].apply(lambda x: isinstance(x, str)).sum()
    
    if string_count > 0:
        print(f"Column '{col}' contains {string_count} string values.")

Column 'financials.annual_income' contains 8 string values.


### 3. Missing or incomplete records

In [5]:
# take care of empty strings by changing them to NaN values
df_clean.replace('', np.nan, inplace=True)

# count total missing values per column
missing_data = df_clean.isnull().sum()
missing_data = missing_data[missing_data > 0]

for column, count in missing_data.items():
    print(f"'{column}': {count} missing values.")

'processing_timestamp': 440 missing values.
'applicant_info.email': 7 missing values.
'applicant_info.ssn': 5 missing values.
'applicant_info.ip_address': 5 missing values.
'applicant_info.gender': 3 missing values.
'applicant_info.date_of_birth': 5 missing values.
'applicant_info.zip_code': 2 missing values.
'financials.annual_income': 5 missing values.
'decision.rejection_reason': 292 missing values.
'loan_purpose': 452 missing values.
'decision.interest_rate': 210 missing values.
'decision.approved_amount': 210 missing values.
'financials.annual_salary': 497 missing values.
'notes': 500 missing values.


### 4. Inconsistent coding/formatting of categorical fields

In [6]:
# check the gender formatting 
gender_counts = df_clean['applicant_info.gender'].value_counts(dropna=False)
print(gender_counts)

# Note that for male there is Male and M, and for female there is Female and F

applicant_info.gender
Male      195
Female    193
F          58
M          53
NaN         3
Name: count, dtype: int64


### 5. Invalid or impossible values

In [7]:
# the credit history cannot be negative 
negative_credit = df_clean[df_clean['financials.credit_history_months'] < 0]

print(f"Found {len(negative_credit)} records with a negative credit history.")
display(negative_credit[['_id', 'financials.credit_history_months']])

# the savings balance cannot be negative
negative_savings = df_clean[df_clean['financials.savings_balance'] < 0]

print(f"Found {len(negative_savings)} records with a negative savings balance.")
display(negative_savings[['_id', 'financials.savings_balance']])

# the processing timestamp cannot be on the future
parsed_timestamps = pd.to_datetime(df_clean['processing_timestamp'], errors='coerce')
current_date = pd.Timestamp.now(tz='UTC')
future_dates = df_clean[parsed_timestamps > current_date]

print(f"Found {len(future_dates)} with processing dates in the future.")
display(future_dates[['_id', 'processing_timestamp']])

# the social security number should be an unique number for each entry/person
ssn_counts = df_clean['applicant_info.ssn'].value_counts()
duplicate_ssns = ssn_counts[ssn_counts > 1]

if not duplicate_ssns.empty:
    print(f"Found {len(duplicate_ssns)} SSN(s) that are shared across multiple records.")
    shared_ssn_records = df_clean[df_clean['applicant_info.ssn'].isin(duplicate_ssns.index)]
    display(shared_ssn_records[['_id', 'applicant_info.full_name', 'applicant_info.ssn']].sort_values(by='applicant_info.ssn'))

# a person needs to be 18+ to ask for a credit
dob_series = pd.to_datetime(df_clean['applicant_info.date_of_birth'], format='mixed', errors='coerce')

age_in_2024 = 2024 - dob_series.dt.year
credit_history_years = df_clean['financials.credit_history_months'] / 12
age_at_credit_start = age_in_2024 - credit_history_years

underage_credit_records = df_clean[age_at_credit_start < 18]

underage_count = len(underage_credit_records)
if underage_count > 0:
    underage_pct = (underage_count / len(df_clean)) * 100
    print(f"Found {underage_count} records ({underage_pct:.2f}%) where credit history begins before age 18.")
    display(underage_credit_records[['_id', 'applicant_info.full_name', 'applicant_info.date_of_birth', 'financials.credit_history_months']].head())

# an approved loan cannot have a rejection reason
illogical_approvals = df_clean[(df_clean['decision.loan_approved'] == True) & (df_clean['decision.rejection_reason'].notnull())]

illogical_rejections = df_clean[(df_clean['decision.loan_approved'] == False) & 
                                (df_clean['decision.approved_amount'].notnull() | df_clean['decision.interest_rate'].notnull())]

print(f"Found {len(illogical_approvals)} approved applications with a rejection reason.")
print(f"Found {len(illogical_rejections)} rejected applications with an approved amount or interest rate.")

if not illogical_approvals.empty:
    display(illogical_approvals[['_id', 'decision.loan_approved', 'decision.rejection_reason']])
if not illogical_rejections.empty:
    display(illogical_rejections[['_id', 'decision.loan_approved', 'decision.approved_amount', 'decision.interest_rate']])

# malformed email addresses
email_pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'
invalid_emails = df_clean[~df_clean['applicant_info.email'].str.match(email_pattern, na=True)]

print(f"\nFound {len(invalid_emails)} records with malformed email addresses.")
if not invalid_emails.empty:
    display(invalid_emails[['_id', 'applicant_info.full_name', 'applicant_info.email']])

# ZIP code length assessment
zip_strings = df_clean['applicant_info.zip_code'].astype(str)
invalid_zips = df_clean[(zip_strings.str.len() != 5) & (df_clean['applicant_info.zip_code'].notnull())]

print(f"Found {len(invalid_zips)} records with invalid ZIP code lengths.")
if not invalid_zips.empty:
    display(invalid_zips[['_id', 'applicant_info.zip_code']])

Found 2 records with a negative credit history.


,_id,financials.credit_history_months
137,app_043,-10
162,app_156,-3


Found 1 records with a negative savings balance.


,_id,financials.savings_balance
159,app_290,-5000


Found 2 with processing dates in the future.


...

Found 2 records (0.40%) where credit history begins before age 18.


,_id,applicant_info.full_name,applicant_info.date_of_birth,financials.credit_history_months
348,app_142,Eric Lee,10/24/2001,66
496,app_049,Donna Gonzalez,2000/05/22,92


Found 0 approved applications with a rejection reason.
Found 0 rejected applications with an approved amount or interest rate.

Found 4 records with malformed email addresses.


,_id,applicant_info.full_name,applicant_info.email
138,app_204,Jonathan Carter,mike johnson@gmail.com
181,app_299,Samuel Gonzalez,test.user.outlook.com
276,app_068,Emily Lopez,john.doe@invalid
369,app_146,Amy Flores,sarah.smith@


Found 0 records with invalid ZIP code lengths.


### 6. Inconsistent date formats

In [8]:
# let's check parsed dates and "failed" dates
parsed_dates = pd.to_datetime(df_clean['applicant_info.date_of_birth'], errors='coerce')
failed_dates = df_clean[parsed_dates.isnull() & df_clean['applicant_info.date_of_birth'].notnull()]

format_issues_count = len(failed_dates)
print(f"Found {format_issues_count} records with inconsistent date formats.")
display(failed_dates[['_id', 'applicant_info.date_of_birth']].head())

# Note that the default pandas date format uses a '-' as a separator, instead of '/'

Found 157 records with inconsistent date formats.


,_id,applicant_info.date_of_birth
5,app_275,14/02/1982
6,app_099,28/01/1990
11,app_320,01/12/1978
14,app_307,1990/07/26
21,app_173,18/07/1979


# Pipeline Code

In [9]:
def clean_credit_data(df_raw):
    df = df_raw.copy()

    # remove the duplicate records, keeping the first instance
    df = df.drop_duplicates(subset=['_id'], keep='first')

    # fix data type of the annual income column
    df['financials.annual_income'] = pd.to_numeric(df['financials.annual_income'], errors='coerce')
    
    # fill missing 'annual_income' with values from 'annual_salary', and drop the latter 
    df['financials.annual_income'] = df['financials.annual_income'].fillna(df['financials.annual_salary'])
    df = df.drop(columns=['financials.annual_salary', 'notes'])

    # map 'M' and 'F' to the standard 'Male' and 'Female'
    gender_mapping = {'M': 'Male', 'F': 'Female'}
    df['applicant_info.gender'] = df['applicant_info.gender'].replace(gender_mapping)

    # fix negative credit history, assuming a data entry typo, we take the absolute value
    df['financials.credit_history_months'] = df['financials.credit_history_months'].abs()

    # fix the negative savings balance, clipping the floor to 0
    df['financials.savings_balance'] = df['financials.savings_balance'].clip(lower=0)

    # fix future timestamps, set future dates to NaT (Not a Time)
    df['processing_timestamp'] = pd.to_datetime(df['processing_timestamp'], errors='coerce')
    current_date = pd.Timestamp.now(tz='UTC')
    df.loc[df['processing_timestamp'] > current_date, 'processing_timestamp'] = pd.NaT

    # parse dates using 'mixed' format so it considers both '-' and '/'
    df['applicant_info.date_of_birth'] = pd.to_datetime(
        df['applicant_info.date_of_birth'],
        format='mixed',
        errors='coerce'
    )

    # drop all records that share an social security number
    df = df.drop_duplicates(subset=['applicant_info.ssn'], keep=False)

    # Calculate age at credit start and filter out the invalid ones
    dob_dt = pd.to_datetime(df['applicant_info.date_of_birth'], format='mixed', errors='coerce')
    age_at_start = (2024 - dob_dt.dt.year) - (df['financials.credit_history_months'] / 12)

    df = df[(age_at_start >= 18) | (age_at_start.isnull())]

    # drop un-auditable records
    df = df.dropna(subset=['applicant_info.gender', 'applicant_info.date_of_birth'])

    return df

# execute the pipeline and get the final dataframe
df_final = clean_credit_data(df_clean)
display(df_final.head())

,_id,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,financials.annual_income,...,spending_Fitness,spending_Gambling,spending_Groceries,spending_Healthcare,spending_Insurance,spending_Rent,spending_Shopping,spending_Transportation,spending_Travel,spending_Utilities
0,app_200,2024-01-15 00:00:00+00:00,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000.0,...,0.0,0.0,0.0,0.0,0.0,790.0,480.0,0.0,0.0,0.0
1,app_037,NaT,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032,78000.0,...,0.0,0.0,0.0,243.0,0.0,608.0,0.0,0.0,0.0,0.0
2,app_215,NaT,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000.0,...,0.0,0.0,0.0,0.0,0.0,109.0,0.0,0.0,0.0,0.0
3,app_024,NaT,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,103000.0,...,575.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,app_184,2024-01-15 00:00:00+00:00,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080,57000.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Export the clean data for next steps

In [10]:
# define the export path
export_path = '../data/clean_credit_applications.csv'

# export the dataframe to CSV
df_final.to_csv(export_path, index=False)